<a href="https://colab.research.google.com/github/thomaslu678/Praxis-Lab-25-26/blob/main/clean/2_ee_task_lst_all_points.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NOTE: Requires points.csv (list of points within digitized buffer zone)
# Assumes columns are lat, long, distance

In [1]:
#@title Copyright 2019 Google LLC. { display-mode: "form" }
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table class="ee-notebook-buttons" align="left"><td>
<a target="_blank"  href="http://colab.research.google.com/github/google/earthengine-community/blob/master/guides/linked/ee-api-colab-setup.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" /> Run in Google Colab</a>
</td><td>
<a target="_blank"  href="https://github.com/google/earthengine-community/blob/master/guides/linked/ee-api-colab-setup.ipynb"><img width=32px src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" /> View source on GitHub</a></td></table>

# Earth Engine Python API Colab Setup

This notebook demonstrates how to setup the Earth Engine Python API in Colab and provides several examples of how to print and visualize Earth Engine processed data.

## Import API and get credentials

The Earth Engine API is installed by default in Google Colaboratory so requires only importing and authenticating. These steps must be completed for each new Colab session, if you restart your Colab kernel, or if your Colab virtual machine is recycled due to inactivity.

### Import the API

Run the following cell to import the API into your session.

In [2]:
import ee

### Authenticate and initialize

Run the `ee.Authenticate` function to authenticate your access to Earth Engine servers and `ee.Initialize` to initialize it. Upon running the following cell you'll be asked to grant Earth Engine access to your Google account. Follow the instructions printed to the cell.

In [3]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='gee-481701')

## Test the API

Test the API by printing the elevation of Mount Everest.

In [4]:

# Print the elevation of Mount Everest.
dem = ee.Image('USGS/SRTMGL1_003')
xy = ee.Geometry.Point([86.9250, 27.9881])
elev = dem.sample(xy, 30).first().get('elevation').getInfo()
print('Mount Everest elevation (m):', elev)

Mount Everest elevation (m): 8729


# Define Areas of Interest

In [5]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
from datetime import timedelta
import scipy.stats as stats
import rasterio
from rasterio.transform import from_origin
from rasterio.features import rasterize
import geopandas as gpd
from shapely.geometry import Point
import requests

In [ ]:
landsat_collections = {
    "L4": {
        "collection": "LANDSAT/LT04/C02/T1_L2",
        "sr": ["SR_B1", "SR_B2", "SR_B3", "SR_B4"],
        "lst": "ST_B6"
    },
    "L5": {
        "collection": "LANDSAT/LT05/C02/T1_L2",
        "sr": ["SR_B1", "SR_B2", "SR_B3", "SR_B4"],
        "lst": "ST_B6"
    },
    "L7": {
        "collection": "LANDSAT/LE07/C02/T1_L2",
        "sr": ["SR_B1", "SR_B2", "SR_B3", "SR_B4"],
        "lst": "ST_B6"
    },
    "L8": {
        "collection": "LANDSAT/LC08/C02/T1_L2",
        "sr": ["SR_B2", "SR_B3", "SR_B4", "SR_B5"],
        "lst": "ST_B10"
    },
    "L9": {
        "collection": "LANDSAT/LC09/C02/T1_L2",
        "sr": ["SR_B2", "SR_B3", "SR_B4", "SR_B5"],
        "lst": "ST_B10"
    }
}

# Functions

In [ ]:
def mask_qa_pixel(img: ee.Image):
    # QA_PIXEL is the name of the fetched file that provides info
    # about pixel quality assurance
    qa = img.select("QA_PIXEL")

    # Fetch bits (0/1) describing if an environmental condition
    # is present
    cloud        = qa.bitwiseAnd(1 << 3).neq(0)
    cloud_shadow = qa.bitwiseAnd(1 << 4).neq(0)
    snow         = qa.bitwiseAnd(1 << 5).neq(0)
    cirrus       = qa.bitwiseAnd(1 << 2).neq(0)

    mask = (
        # Or condition for all bad conditions
        cloud
        .Or(cloud_shadow)
        .Or(snow)
        .Or(cirrus)

        # Not flips the conditions so any remaining pixels
        # have none of the conditions, and are "good" quality.
        .Not()
    )

    # Apply the mask to the input image
    return img.updateMask(mask)

In [ ]:
def scale_sr(img, sr_bands):
    scaled = (
        img.select(sr_bands)
        .multiply(2.75e-05)
        .add(-0.2)
        .rename(["Blue", "Green", "Red", "NIR"])
    )
    return img.addBands(scaled)

def scale_lst(img, lst_band):
    lst = (
        img.select(lst_band)
        .multiply(0.00341802)
        .add(149.0)
        .rename("LST_K")
    )
    return img.addBands(lst)


# Loop over all points

In [ ]:
input_df = pd.read_csv('/content/sample_data/2_points_clean.csv') # id column is called id

In [ ]:
# input_df['point_id'] = input_df.index

In [ ]:
input_df

,id,long,lat
0,17,126.974229,37.570410
1,18,126.974231,37.570140
2,19,126.974233,37.569869
3,20,126.974235,37.569599
4,21,126.974236,37.569328
...,...,...,...
4889,7708,127.044881,37.572055
4890,7709,127.044882,37.571785
4891,7710,127.044884,37.571515
4892,7711,127.044886,37.571244


In [ ]:
def df_to_points_fc(df):
    feats = []
    for _, r in df.iterrows():
        geom = ee.Geometry.Point(float(r["long"]), float(r["lat"]))
        feats.append(
            ee.Feature(geom, {
                "point_id": r["id"]
            })
        )
    return ee.FeatureCollection(feats)

points_fc = df_to_points_fc(input_df)

In [ ]:
city_geom = points_fc.geometry().convexHull(maxError=1)

In [ ]:
def sample_landsat_collection(cfg, mission, city_geom, points_fc):

    col = (
        ee.ImageCollection(cfg["collection"])
        .filterBounds(city_geom)
        .map(mask_qa_pixel)
        .map(lambda img: scale_sr(img, cfg["sr"]))
        .map(lambda img: scale_lst(img, cfg["lst"]))
    )

    bands_to_sample = ["Blue", "Green", "Red", "NIR", "LST_K"]

    def per_image(img):
        samples = img.select(bands_to_sample).sampleRegions(
            collection=points_fc,
            geometries=False,
            scale=30
        )

        return samples.map(lambda f: f.set({
            # "mission": mission,
            "time": ee.Date(img.get("system:time_start"))
                    .format("YYYY-MM-dd HH:mm:ss")
        }))

    return col.map(per_image).flatten()

In [ ]:
collections = []

for mission, cfg in landsat_collections.items():
    fc = sample_landsat_collection(cfg, mission, city_geom, points_fc)
    collections.append(fc)

final_fc = ee.FeatureCollection(collections).flatten()

In [ ]:
output_file_name = "2_landsat_clean"

In [ ]:
ee.batch.Export.table.toDrive(
    collection=final_fc,
    description=output_file_name,
    fileFormat="CSV",
    selectors=[
        "point_id",
        "time",
        "Blue",
        "Green",
        "Red",
        "NIR",
        "LST_K"
    ]
).start()

Link to GEE Tasks:

https://code.earthengine.google.com/tasks

In [ ]:
ee.data.getTaskStatus("IVCZD6FL5HJIDAKPJ4DNGIP6")

[{'state': 'READY',
  'description': '2_landsat_clean',
  'priority': 100,
  'creation_timestamp_ms': 1775858942160,
  'update_timestamp_ms': 1775858954665,
  'start_timestamp_ms': 0,
  'task_type': 'EXPORT_FEATURES',
  'id': 'IVCZD6FL5HJIDAKPJ4DNGIP6',
  'name': 'projects/gee-481701/operations/IVCZD6FL5HJIDAKPJ4DNGIP6'}]

In [ ]:
ee.data.listOperations()


[{'name': 'projects/gee-481701/operations/AUEY5NN2M26ROERPJPRJ3TZD',
  'metadata': {'@type': 'type.googleapis.com/google.earthengine.v1alpha.OperationMetadata',
   'state': 'RUNNING',
   'description': '0_landsat_clean',
   'priority': 100,
   'createTime': '2026-04-10T21:50:47.712893Z',
   'updateTime': '2026-04-10T21:54:54.695921Z',
   'startTime': '2026-04-10T21:50:54.277147Z',
   'type': 'EXPORT_FEATURES',
   'attempt': 1,
   'batchEecuUsageSeconds': 560.430786132}},
 {'name': 'projects/gee-481701/operations/L7RC5WUM6IWARPSBZSPFDIDN',
  'metadata': {'@type': 'type.googleapis.com/google.earthengine.v1alpha.OperationMetadata',
   'state': 'SUCCEEDED',
   'description': 'Merged studies',
   'priority': 100,
   'createTime': '2026-04-10T20:50:16.711664Z',
   'updateTime': '2026-04-10T21:29:23.104843Z',
   'startTime': '2026-04-10T20:50:22.175422Z',
   'endTime': '2026-04-10T21:29:23.104843Z',
   'type': 'EXPORT_FEATURES',
   'destinationUris': ['https://code.earthengine.google.com/?ass

# Test

Extent
-80.0109902496288754,40.4392394961659747 : -79.9934291780780740,40.4478472153211897

In [22]:
# Example coordinates
y1, x1 = -33.88409595418479, 151.2000255522259

# directly paste the gmaps coordinate (reverse if from QGIS)
y2, x2 = -33.88075586646993, 151.20291160906882

xmin, ymin, xmax, ymax = min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2)

roi = ee.Geometry.Rectangle(
    [xmin, ymin, xmax, ymax],
    proj=None,
    geodesic=False
)

In [23]:
collections = [
    "LANDSAT/LT04/C02/T1_L2",
    "LANDSAT/LT05/C02/T1_L2",
    "LANDSAT/LE07/C02/T1_L2",
    "LANDSAT/LC08/C02/T1_L2",
    "LANDSAT/LC09/C02/T1_L2"
]

merged = ee.ImageCollection(
    ee.ImageCollection(collections[0])
)

In [24]:
for c in collections[1:]:
    merged = merged.merge(
        ee.ImageCollection(c)
    )

img = (
    merged
    .filterBounds(roi)
    .sort("system:time_start")
    .first()
)

print("Image ID:")
print(img.getInfo()["id"])

Image ID:
LANDSAT/LT05/C02/T1_L2/LT05_089083_19870522


In [25]:
props = img.toDictionary().getInfo()

print("Number of properties:", len(props))

for k, v in props.items():
    print(k, v)

Number of properties: 99
ALGORITHM_SOURCE_SURFACE_REFLECTANCE LEDAPS_3.4.0
ALGORITHM_SOURCE_SURFACE_TEMPERATURE st_1.3.0
CLOUD_COVER 1
CLOUD_COVER_LAND 0
COLLECTION_CATEGORY T1
COLLECTION_NUMBER 2
CORRECTION_BIAS_BAND_1 CPF
CORRECTION_BIAS_BAND_2 CPF
CORRECTION_BIAS_BAND_3 CPF
CORRECTION_BIAS_BAND_4 CPF
CORRECTION_BIAS_BAND_5 CPF
CORRECTION_BIAS_BAND_6 CPF
CORRECTION_BIAS_BAND_7 CPF
CORRECTION_GAIN_BAND_1 CPF
CORRECTION_GAIN_BAND_2 CPF
CORRECTION_GAIN_BAND_3 CPF
CORRECTION_GAIN_BAND_4 CPF
CORRECTION_GAIN_BAND_5 CPF
CORRECTION_GAIN_BAND_6 INTERNAL_CALIBRATION
CORRECTION_GAIN_BAND_7 CPF
DATA_SOURCE_AIR_TEMPERATURE NCEP
DATA_SOURCE_ELEVATION GLS2000
DATA_SOURCE_OZONE TOMS
DATA_SOURCE_PRESSURE NCEP
DATA_SOURCE_REANALYSIS MERRA-2
DATA_SOURCE_WATER_VAPOR NCEP
DATA_TYPE_L0RP TMR_L0RP
DATE_ACQUIRED 1987-05-22
DATE_PRODUCT_GENERATED 1602706117000
DATUM WGS84
EARTH_SUN_DISTANCE 1.0124129
ELLIPSOID WGS84
EPHEMERIS_TYPE DEFINITIVE
GEOMETRIC_RMSE_MODEL 4.835
GEOMETRIC_RMSE_MODEL_X 3.467
GEOMETRIC_R

In [26]:
band_info = img.bandNames().getInfo()

print("Bands:")
for b in band_info:
    print(b)

Bands:
SR_B1
SR_B2
SR_B3
SR_B4
SR_B5
SR_B7
SR_ATMOS_OPACITY
SR_CLOUD_QA
ST_B6
ST_ATRAN
ST_CDIST
ST_DRAD
ST_EMIS
ST_EMSD
ST_QA
ST_TRAD
ST_URAD
QA_PIXEL
QA_RADSAT


In [27]:
proj = img.select(0).projection()

coords = ee.Image.pixelCoordinates(proj)

img_with_coords = img.addBands(coords)

In [28]:
samples = img_with_coords.sample(
    region=roi,
    scale=30,
    geometries=True
)

In [19]:
features = samples.getInfo()["features"]

records = []

for f in features:

    row = f["properties"].copy()

    row["lon"] = f["geometry"]["coordinates"][0]
    row["lat"] = f["geometry"]["coordinates"][1]

    records.append(row)

df = pd.DataFrame(records)

In [ ]:
df.iloc[0][['x', 'y']]

,0
x,2486.5
y,3075.5


In [ ]:
df.iloc[1][['x', 'y']]

,1
x,2487.5
y,3075.5


In [29]:
proj_info = img.select(0).projection().getInfo()

print(proj_info)

{'type': 'Projection', 'crs': 'EPSG:32656', 'transform': [30, 0, 261585, 0, -30, -3565785]}


In [ ]:
df['x'].min()

2443.5

In [ ]:
df['y'].min()

3075.5

# Quick Pull Reconstruction

In [93]:
landsat_collections = {
    "L4": {
        "collection": "LANDSAT/LT04/C02/T1_L2",
        "sr": ["SR_B1", "SR_B2", "SR_B3", "SR_B4"],
        "lst": "ST_B6"
    },
    "L5": {
        "collection": "LANDSAT/LT05/C02/T1_L2",
        "sr": ["SR_B1", "SR_B2", "SR_B3", "SR_B4"],
        "lst": "ST_B6"
    },
    "L7": {
        "collection": "LANDSAT/LE07/C02/T1_L2",
        "sr": ["SR_B1", "SR_B2", "SR_B3", "SR_B4"],
        "lst": "ST_B6"
    },
    "L8": {
        "collection": "LANDSAT/LC08/C02/T1_L2",
        "sr": ["SR_B2", "SR_B3", "SR_B4", "SR_B5"],
        "lst": "ST_B10"
    },
    "L9": {
        "collection": "LANDSAT/LC09/C02/T1_L2",
        "sr": ["SR_B2", "SR_B3", "SR_B4", "SR_B5"],
        "lst": "ST_B10"
    }
}

In [94]:
points_df = pd.read_csv("/content/0_points.csv")

In [95]:
def df_to_points_fc(df):

    feats = []

    for _, r in df.iterrows():

        feat = ee.Feature(
            ee.Geometry.Point([float(r["x"]), float(r["y"])]),
            {
                "fid": int(r["fid"]),
                "row_index": int(r["row_index"]),
                "col_index": int(r["col_index"])
            }
        )

        feats.append(feat)

    return ee.FeatureCollection(feats)

In [96]:
def mask_qa_pixel(img):

    qa = img.select("QA_PIXEL")

    cloud = qa.bitwiseAnd(1 << 3).neq(0)
    cloud_shadow = qa.bitwiseAnd(1 << 4).neq(0)
    snow = qa.bitwiseAnd(1 << 5).neq(0)
    cirrus = qa.bitwiseAnd(1 << 2).neq(0)

    mask = (
        cloud
        .Or(cloud_shadow)
        .Or(snow)
        .Or(cirrus)
        .Not()
    )

    return img.updateMask(mask)

In [97]:
def scale_sr(img, sr_bands):

    scaled = (
        img.select(sr_bands)
        .multiply(2.75e-05)
        .add(-0.2)
        .rename(["Blue", "Green", "Red", "NIR"])
    )

    return img.addBands(scaled)

In [98]:
def scale_lst(img, lst_band):

    lst = (
        img.select(lst_band)
        .multiply(0.00341802)
        .add(149.0)
        .rename("LST_K")
    )

    return img.addBands(lst)

In [99]:
points_fc = df_to_points_fc(points_df)

In [100]:
city_geom = points_fc.geometry().convexHull(maxError=1)

In [101]:
fid_img = (
    points_fc.reduceToImage(
        properties=["fid"],
        reducer=ee.Reducer.first()
    )
    .rename("fid")
)

row_img = (
    points_fc.reduceToImage(
        properties=["row_index"],
        reducer=ee.Reducer.first()
    )
    .rename("row_index")
)

col_img = (
    points_fc.reduceToImage(
        properties=["col_index"],
        reducer=ee.Reducer.first()
    )
    .rename("col_index")
)

grid_info = fid_img.addBands(row_img).addBands(col_img)

In [102]:
def process_collection(
    cfg,
    mission,
    region,
    start_year,
    end_year
):

    start_date = ee.Date.fromYMD(
        start_year,
        1,
        1
    )

    end_date_exclusive = ee.Date.fromYMD(
        end_year + 1,
        1,
        1
    )

    col = (
        ee.ImageCollection(cfg["collection"])
        .filterBounds(region)
        .filterDate(
            start_date,
            end_date_exclusive
        )
        .map(mask_qa_pixel)
        .map(lambda img: scale_sr(img, cfg["sr"]))
        .map(lambda img: scale_lst(img, cfg["lst"]))
    )

    return col

In [103]:
def image_metadata_fc(col):

    def make_feature(img):

        return ee.Feature(
            None,
            {
                "mission_id": img.id(),

                "time":
                    ee.Date(
                        img.get("system:time_start")
                    ).format("YYYY-MM-dd HH:mm:ss"),

                "cloud_cover":
                    img.get("CLOUD_COVER"),

                "sun_azimuth":
                    img.get("SUN_AZIMUTH"),

                "sun_elevation":
                    img.get("SUN_ELEVATION")
            }
        )

    return ee.FeatureCollection(col.map(make_feature))

In [104]:
bands_to_keep = [
    "fid",
    "row_index",
    "col_index",
    "Blue",
    "Green",
    "Red",
    "NIR",
    "LST_K"
]

In [105]:
def image_samples(img):

    image_id = img.id()

    stack = (
        grid_info
        .addBands(
            img.select(
                ["Blue", "Green", "Red", "NIR", "LST_K"]
            )
        )
    )

    samples = stack.sample(
        region=city_geom,
        scale=30,
        geometries=False
    )

    return samples.map(
        lambda f: f.set(
            "mission_id",
            image_id
        )
    )

In [106]:
all_samples = []
all_metadata = []

In [107]:
start_year = 1994
end_year = 2001

In [108]:
start_year = 1994
end_year = 2001

for mission, cfg in landsat_collections.items():

    col = process_collection(
        cfg,
        mission,
        city_geom,
        start_year,
        end_year
    )

    metadata_fc = image_metadata_fc(col)

    samples_fc = ee.FeatureCollection(
        col.map(image_samples)
    ).flatten()

    all_samples.append(samples_fc)
    all_metadata.append(metadata_fc)

In [109]:
final_samples = ee.FeatureCollection(
    all_samples
).flatten()

final_metadata = ee.FeatureCollection(
    all_metadata
).flatten()

In [110]:
ee.batch.Export.table.toDrive(
    collection=final_samples,
    description="landsat_pixel_values",
    fileFormat="CSV",
    selectors=[
        "fid",
        "row_index",
        "col_index",
        "mission_id",
        "Blue",
        "Green",
        "Red",
        "NIR",
        "LST_K"
    ]
).start()

In [111]:
ee.batch.Export.table.toDrive(
    collection=final_metadata,
    description="landsat_image_metadata",
    fileFormat="CSV",
    selectors=[
        "mission_id",
        "time",
        "cloud_cover",
        "sun_azimuth",
        "sun_elevation"
    ]
).start()

## Validation of exports

In [112]:
samples_df = pd.read_csv("/content/landsat_pixel_values.csv")

input_fids = set(points_df["fid"])
output_fids = set(samples_df["fid"])

invalid_fids = output_fids - input_fids

print("Invalid FIDs:", len(invalid_fids))

Invalid FIDs: 0


In [113]:
missing_fids = input_fids - output_fids

print("Points with zero observations:", len(missing_fids))

Points with zero observations: 29


In [114]:
len(points_df)

593

In [115]:
points_df[points_df["fid"].isin(missing_fids)]

,fid,row_index,col_index,distance,x,y
26,27,22,13,56.149940,-80.005580,40.442897
112,113,31,5,335.974383,-80.008446,40.440489
142,143,28,2,333.070363,-80.009495,40.441309
157,158,27,1,342.765557,-80.009844,40.441582
160,161,15,45,371.077090,-79.994234,40.444691
161,162,6,44,342.695540,-79.994551,40.447126
166,167,16,43,327.450676,-79.994945,40.444427
168,169,15,43,314.947620,-79.994941,40.444697
179,180,12,45,348.428313,-79.994222,40.445501
180,181,11,45,345.753972,-79.994218,40.445772


In [116]:
dupes = samples_df.duplicated(
    subset=["fid", "mission_id"]
)

print("Duplicate fid/image pairs:", dupes.sum())

Duplicate fid/image pairs: 0


In [117]:
metadata_df = pd.read_csv(
    "/content/landsat_image_metadata.csv"
)

sample_images = set(samples_df["mission_id"])
metadata_images = set(metadata_df["mission_id"])

missing_metadata = (
    sample_images - metadata_images
)

print(
    "Images referenced by samples "
    "but absent from metadata:",
    len(missing_metadata)
)

Images referenced by samples but absent from metadata: 0


In [118]:
print(
    metadata_df["mission_id"]
    .duplicated()
    .sum()
)

0


In [119]:
counts = (
    metadata_df
    .groupby("mission_id")
    .size()
)

print(counts.describe())

count    70.0
mean      1.0
std       0.0
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       1.0
dtype: float64


In [120]:
bands = [
    "Blue",
    "Green",
    "Red",
    "NIR",
    "LST_K"
]

all_null = (
    samples_df[bands]
    .isna()
    .all(axis=1)
)

print(
    "Rows with all bands null:",
    all_null.sum()
)

Rows with all bands null: 0


In [121]:
for b in ["Blue","Green","Red","NIR"]:
    print()
    print(b)
    print(samples_df[b].describe())


Blue
count    16708.000000
mean         0.053727
std          0.044464
min         -0.198047
25%          0.028002
50%          0.051199
75%          0.077200
max          0.378490
Name: Blue, dtype: float64

Green
count    16708.000000
mean         0.079165
std          0.049455
min         -0.121048
25%          0.045685
50%          0.076334
75%          0.107539
max          0.395622
Name: Green, dtype: float64

Red
count    16708.000000
mean         0.084462
std          0.057501
min         -0.199807
25%          0.045843
50%          0.081119
75%          0.119027
max          0.427522
Name: Red, dtype: float64

NIR
count    16708.000000
mean         0.142752
std          0.092397
min         -0.155450
25%          0.076595
50%          0.134813
75%          0.195807
max          0.542912
Name: NIR, dtype: float64


In [122]:
samples_df["LST_K"].describe()

,LST_K
count,16708.000000
mean,298.275376
std,14.532605
min,258.089526
25%,288.714131
50%,302.260599
75%,307.986637
max,326.196993


In [123]:
print(
    "Unique images in samples:",
    samples_df["mission_id"].nunique()
)

print(
    "Rows in metadata:",
    len(metadata_df)
)

Unique images in samples: 42
Rows in metadata: 70


In [124]:
is_subset = samples_df["mission_id"].isin(metadata_df['mission_id']).all()
print(is_subset)

True


In [125]:
input_lookup = (
    points_df
    .set_index("fid")
    [["row_index","col_index"]]
)

merged = samples_df.merge(
    input_lookup,
    left_on="fid",
    right_index=True,
    suffixes=("_export","_input")
)

bad_rows = merged[
    (
        merged["row_index_export"]
        !=
        merged["row_index_input"]
    )
    |
    (
        merged["col_index_export"]
        !=
        merged["col_index_input"]
    )
]

print(
    "Row/col mismatches:",
    len(bad_rows)
)

Row/col mismatches: 0
